In [27]:
import sys
from pathlib import Path

# Add the repo root / src folder to sys.path
repo_root = Path.cwd()  # notebook current folder is notebooks/
sys.path.append(str(repo_root.parent / "src"))

In [28]:
import os
os.getcwd()

'c:\\Users\\User\\Documents\\retailmind-prototype\\notebooks'

In [29]:
from ingestion.load_ccpe import load_ccpe, quick_stats
import pandas as pd

In [30]:
ccpe_raw_path = "../data/raw/ccpe.txt"   # path to the raw CCPE data
df_ccpe = load_ccpe(ccpe_raw_path)       # load the CCPE data
quick_stats(df_ccpe)     # print some quick stats

Num rows: 12436
Num conversations: 500
Num OVERALL lines: 500
Speakers: {'USER': 6860, 'SYSTEM': 5576}


In [31]:
df_ccpe.head(10)      #sample 10 rows

,conv_id,turn_id,speaker,text,entity_tag,scores_raw,scores,is_overall
0,0,1,SYSTEM,Do you like movies like Thor?,ENTITY_NAME+MOVIE_OR_SERIES,None,None,False
1,0,2,USER,"No, I don't like Thor.",ENTITY_NAME+MOVIE_OR_SERIES,"3,2,2","[3, 2, 2]",False
2,0,3,SYSTEM,Ok. What is it about this type of movie that y...,OTHER,None,None,False
3,0,4,USER,I don't like all the,OTHER,"3,2,2","[3, 2, 2]",False
4,0,5,USER,Like the I don't know. Like is it voice acting?,OTHER,"3,2,2","[3, 2, 2]",False
5,0,6,USER,in that controls the characters,OTHER,"3,3,2","[3, 3, 2]",False
6,0,7,SYSTEM,Can you say a little more about that please?,OTHER,None,None,False
7,0,8,USER,"Yeah, so I don't know I guess the acting was bad.",ENTITY_PREFERENCE+MOVIE_OR_SERIES,"3,3,2","[3, 3, 2]",False
8,0,9,USER,I was wondering if it's voice acting that cont...,OTHER,"3,3,3","[3, 3, 3]",False
9,0,10,SYSTEM,Do you like movies like Star Wars?,ENTITY_NAME+MOVIE_OR_SERIES,None,None,False


In [32]:
df_ccpe.info()        # get info on dataframe columns and types

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12436 entries, 0 to 12435
Data columns (total 8 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   conv_id     12436 non-null  int64 
 1   turn_id     12436 non-null  int64 
 2   speaker     12436 non-null  object
 3   text        12436 non-null  object
 4   entity_tag  12436 non-null  object
 5   scores_raw  6860 non-null   object
 6   scores      6860 non-null   object
 7   is_overall  12436 non-null  bool  
dtypes: bool(1), int64(2), object(5)
memory usage: 692.4+ KB


In [33]:
df_ccpe["entity_tag"].value_counts().to_dict()   # get counts of unique entity tags

{'OTHER': 5615,
 'ENTITY_NAME+MOVIE_OR_SERIES': 2132,
 'ENTITY_PREFERENCE+MOVIE_OR_SERIES': 1924,
 'ENTITY_PREFERENCE+MOVIE_GENRE_OR_CATEGORY': 1440,
 'ENTITY_OTHER+MOVIE_OR_SERIES': 734,
 'ENTITY_NAME+MOVIE_GENRE_OR_CATEGORY': 282,
 'ENTITY_DESCRIPTION+MOVIE_OR_SERIES': 95,
 'ENTITY_PREFERENCE+PERSON': 74,
 'ENTITY_DESCRIPTION+MOVIE_GENRE_OR_CATEGORY': 47,
 'ENTITY_OTHER+MOVIE_GENRE_OR_CATEGORY': 30,
 'ENTITY_NAME+SOMETHING_ELSE': 20,
 'ENTITY_NAME+PERSON': 14,
 'ENTITY_OTHER+SOMETHING_ELSE': 11,
 'ENTITY_PREFERENCE+SOMETHING_ELSE': 10,
 'ENTITY_OTHER+PERSON': 6,
 'ENTITY_DESCRIPTION+PERSON': 2}

In [34]:
def avg_score(score_list):
    if not score_list:
        return None
    return sum(score_list) / len(score_list)

df_ccpe["satisfaction_mean"] = df_ccpe["scores"].apply(avg_score)
df_ccpe.head(10)  # show the updated dataframe with mean satisfaction score

,conv_id,turn_id,speaker,text,entity_tag,scores_raw,scores,is_overall,satisfaction_mean
0,0,1,SYSTEM,Do you like movies like Thor?,ENTITY_NAME+MOVIE_OR_SERIES,None,None,False,NaN
1,0,2,USER,"No, I don't like Thor.",ENTITY_NAME+MOVIE_OR_SERIES,"3,2,2","[3, 2, 2]",False,2.333333
2,0,3,SYSTEM,Ok. What is it about this type of movie that y...,OTHER,None,None,False,NaN
3,0,4,USER,I don't like all the,OTHER,"3,2,2","[3, 2, 2]",False,2.333333
4,0,5,USER,Like the I don't know. Like is it voice acting?,OTHER,"3,2,2","[3, 2, 2]",False,2.333333
5,0,6,USER,in that controls the characters,OTHER,"3,3,2","[3, 3, 2]",False,2.666667
6,0,7,SYSTEM,Can you say a little more about that please?,OTHER,None,None,False,NaN
7,0,8,USER,"Yeah, so I don't know I guess the acting was bad.",ENTITY_PREFERENCE+MOVIE_OR_SERIES,"3,3,2","[3, 3, 2]",False,2.666667
8,0,9,USER,I was wondering if it's voice acting that cont...,OTHER,"3,3,3","[3, 3, 3]",False,3.000000
9,0,10,SYSTEM,Do you like movies like Star Wars?,ENTITY_NAME+MOVIE_OR_SERIES,None,None,False,NaN


In [35]:
from  ingestion.load_data import quick_stats
quick_stats(df_ccpe)

Num rows: 12436
Num conversations: 500
Num OVERALL lines: 500
Speakers: {'USER': 6860, 'SYSTEM': 5576}


In [36]:
df_ccpe['label']=df_ccpe['entity_tag']

In [37]:
# Saving processed CCPE
ccpe_processed_path = "../data/processed/ccpe_turns.parquet"
df_ccpe.to_parquet(ccpe_processed_path, index=False)
print(f"CCPE processed and saved to: {ccpe_processed_path}")

CCPE processed and saved to: ../data/processed/ccpe_turns.parquet


In [38]:
import pandas as pd
df_ccpe = pd.read_parquet("../data/processed/ccpe_turns.parquet")
df_ccpe.head(10)    

,conv_id,turn_id,speaker,text,entity_tag,scores_raw,scores,is_overall,satisfaction_mean,label
0,0,1,SYSTEM,Do you like movies like Thor?,ENTITY_NAME+MOVIE_OR_SERIES,None,None,False,NaN,ENTITY_NAME+MOVIE_OR_SERIES
1,0,2,USER,"No, I don't like Thor.",ENTITY_NAME+MOVIE_OR_SERIES,"3,2,2","[3, 2, 2]",False,2.333333,ENTITY_NAME+MOVIE_OR_SERIES
2,0,3,SYSTEM,Ok. What is it about this type of movie that y...,OTHER,None,None,False,NaN,OTHER
3,0,4,USER,I don't like all the,OTHER,"3,2,2","[3, 2, 2]",False,2.333333,OTHER
4,0,5,USER,Like the I don't know. Like is it voice acting?,OTHER,"3,2,2","[3, 2, 2]",False,2.333333,OTHER
5,0,6,USER,in that controls the characters,OTHER,"3,3,2","[3, 3, 2]",False,2.666667,OTHER
6,0,7,SYSTEM,Can you say a little more about that please?,OTHER,None,None,False,NaN,OTHER
7,0,8,USER,"Yeah, so I don't know I guess the acting was bad.",ENTITY_PREFERENCE+MOVIE_OR_SERIES,"3,3,2","[3, 3, 2]",False,2.666667,ENTITY_PREFERENCE+MOVIE_OR_SERIES
8,0,9,USER,I was wondering if it's voice acting that cont...,OTHER,"3,3,3","[3, 3, 3]",False,3.000000,OTHER
9,0,10,SYSTEM,Do you like movies like Star Wars?,ENTITY_NAME+MOVIE_OR_SERIES,None,None,False,NaN,ENTITY_NAME+MOVIE_OR_SERIES


In [39]:
df_ccpe["label"].value_counts()

label
OTHER                                         5615
ENTITY_NAME+MOVIE_OR_SERIES                   2132
ENTITY_PREFERENCE+MOVIE_OR_SERIES             1924
ENTITY_PREFERENCE+MOVIE_GENRE_OR_CATEGORY     1440
ENTITY_OTHER+MOVIE_OR_SERIES                   734
ENTITY_NAME+MOVIE_GENRE_OR_CATEGORY            282
ENTITY_DESCRIPTION+MOVIE_OR_SERIES              95
ENTITY_PREFERENCE+PERSON                        74
ENTITY_DESCRIPTION+MOVIE_GENRE_OR_CATEGORY      47
ENTITY_OTHER+MOVIE_GENRE_OR_CATEGORY            30
ENTITY_NAME+SOMETHING_ELSE                      20
ENTITY_NAME+PERSON                              14
ENTITY_OTHER+SOMETHING_ELSE                     11
ENTITY_PREFERENCE+SOMETHING_ELSE                10
ENTITY_OTHER+PERSON                              6
ENTITY_DESCRIPTION+PERSON                        2
Name: count, dtype: int64